# Cryptic IP Binding Sites — Full Pipeline (Google Colab)

Run the complete post-Phase-1 workflow on Colab:

1. Install fpocket, APBS, and Python deps
2. Tier-1 validation gate (ADAR2 vs PLCδ1)
3. ML retrain with pocket-level labels
4. Yeast pilot screen (configurable N proteins)
5. Publication package + optional MD pilot

**Runtime:** Use a **CPU** or **GPU** runtime (GPU not required). For 500 proteins, allow **2–4 hours**.

**Tip:** Start with `N_PROTEINS = 25` to verify, then increase to 500.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = 'cryptic-ip-binding-sites'
BRANCH = 'cursor/full-pipeline-improvements-6dd9'  # or 'main'

if IN_COLAB:
    if not Path(REPO).exists():
        !git clone -b {BRANCH} https://github.com/Tommaso-R-Marena/cryptic-ip-binding-sites.git {REPO}
    os.chdir(REPO)
    !bash scripts/colab_install.sh
else:
    os.chdir(Path('..').resolve() if Path('cryptic_ip').exists() is False and Path('../cryptic_ip').exists() else Path('.'))

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print('Working directory:', ROOT)

In [ ]:
# --- Configuration ---
N_PROTEINS = 25          # increase to 500 for full yeast pilot
WORKERS = 2              # Colab typically has 2 CPUs
WITH_ELECTROSTATICS = False  # True adds APBS per structure (much slower)
RUN_ML = True
RUN_YEAST = True
RUN_PUBLICATION = True
RUN_MD_PILOT = False     # needs OpenMM: pip install openmm mdtraj

OUTPUT = ROOT / 'results' / 'colab_run'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Output:', OUTPUT)

## 1. Tier-1 validation gate

In [ ]:
from cryptic_ip.validation.validation_suite import ValidationSuite

suite = ValidationSuite(data_dir='data/validation', use_electrostatics=WITH_ELECTROSTATICS)
summary = suite.run_full_validation(output_dir=OUTPUT / 'validation')
sep = summary['separation_quality']
print('Phase 1 ready:', sep.get('phase1_ready'))
print('Tier-1 separation:', sep.get('tier1_separation'))

## 2. ML retrain (pocket-level labels, grouped by PDB)

In [ ]:
if RUN_ML:
    import subprocess
    cmd = [
        sys.executable, 'scripts/train_ml_classifier.py',
        '--skip-build-dataset',
        '--work-dir', str(OUTPUT / 'ml_training'),
        '--model-dir', str(ROOT / 'models'),
    ]
    if WITH_ELECTROSTATICS:
        cmd.append('--include-electrostatics')
    subprocess.run(cmd, check=True)
    import pandas as pd
    display(pd.read_csv(OUTPUT / 'ml_training' / 'ml_vs_threshold_comparison.csv'))

## 3. Yeast pilot screen

In [ ]:
if RUN_YEAST:
    import subprocess
    cmd = [
        sys.executable, 'scripts/run_yeast_pilot_screen.py',
        '--n-proteins', str(N_PROTEINS),
        '--workers', str(WORKERS),
        '--output-dir', str(OUTPUT / 'yeast_pilot'),
        '--structures-dir', str(ROOT / 'data' / 'structures' / 'yeast_pilot'),
    ]
    if WITH_ELECTROSTATICS:
        cmd.append('--with-electrostatics')
    else:
        cmd.append('--skip-electrostatics')
    subprocess.run(cmd, check=True)
    import json
    print(json.dumps(json.loads((OUTPUT / 'yeast_pilot' / 'yeast_pilot_summary.json').read_text()), indent=2))

## 4. Publication package

In [ ]:
if RUN_PUBLICATION:
    import subprocess
    cmd = [
        sys.executable, 'scripts/run_publication_package.py',
        '--output-dir', str(OUTPUT / 'publication'),
        '--skip-dataset-build',
        '--skip-figures',
    ]
    if WITH_ELECTROSTATICS:
        cmd.append('--with-electrostatics')
    subprocess.run(cmd, check=True)
    from IPython.display import Markdown
    display(Markdown((OUTPUT / 'publication' / 'RESULTS_SUMMARY.md').read_text()))

## 5. Optional MD pilot (OpenMM required)

In [ ]:
if RUN_MD_PILOT:
    !pip install -q openmm mdtraj
    import subprocess
    candidates = OUTPUT / 'publication' / 'gallery' / 'gallery_inputs.csv'
    if not candidates.exists():
        candidates = ROOT / 'results' / 'publication' / 'gallery' / 'gallery_inputs.csv'
    subprocess.run([
        sys.executable, 'scripts/run_md_pilot_validation.py',
        '--candidates-csv', str(candidates),
        '--output-dir', str(OUTPUT / 'md_validation'),
        '--top-n', '3',
        '--production-ns', '0.5',
    ], check=False)

## 6. Download results (Colab only)

In [ ]:
if IN_COLAB:
    !zip -r colab_pipeline_results.zip results/colab_run
    from google.colab import files
    files.download('colab_pipeline_results.zip')